In [1]:
import builtins
from datetime import datetime

_original_print = builtins.print

def print(*args, **kwargs):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    _original_print(f"[{timestamp}]", *args, **kwargs)

builtins.print = print

In [2]:
#Python Modules
import os
import joblib
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
import torch 
from torch.utils.data import DataLoader

os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
os.environ["TORCH_USE_CUDA_DSA"]   = "1"

#ML Modules
import train.train_code as tr_c
import preprocess.data_frame_preprocess as prep
import models.model_bge_m3 as bge_m3
import models.model_xgboost as xgb_m3
import evaluate
import dataset.Train as multi_task

#Config FILE
import config_test as cfg

/home/andreoseo/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Transformer

## Check Save Paths

In [3]:
os.makedirs(cfg.paths.SAVE_DIR,exist_ok=True)

In [4]:
raw_df = pd.read_csv(cfg.paths.CSV_PATH)
df = prep.prepare_dataframe(raw_df,cfg) #New columns: lyrics_clean & gloss_clean

strat_key = (df['label_t1'].astype(str) + '_' +
             df['label_t3'].astype(str) + '_' +
             df[cfg.columns.LABEL_COLS_T2].astype(str).agg('-'.join, axis=1))

[2026-05-05 11:30] [METRICS] Palabras promedio PRE:  338.54
[2026-05-05 11:30] [METRICS] Palabras promedio POST: 286.44
[2026-05-05 11:30] [METRICS] Reducción de tokens:    15.39%
[2026-05-05 11:30] [FLAG] Usando Glosses para entrenamiento


## Load Models

In [5]:
#Load Transformer Model
device    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer,model = bge_m3.load_tokenizer_model(cfg,device)

Loadin


## [DEFINE] Dataset and Loader

In [6]:
test_ds = multi_task.MultiTaskDataset(df, cfg, tokenizer, cfg.tokenization.MAX_LEN,
                            head_len=cfg.tokenization.HEAD_LEN,
                            model_prefix=cfg.paths.MODEL_PREFIX)

test_loader = DataLoader(test_ds, batch_size=cfg.train.BATCH_SIZE,
                         num_workers=10, pin_memory=True)

# [LOAD] Weights (BGE M3 & XGBOOST)

In [7]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Cargar pesos entrenados
#tokenizer, model = load_tokenizer_model(cfg, device)

In [8]:
model.load_state_dict(torch.load(os.path.join(
            '/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/bg3_deduplicatedn_t1',
            'model_weights.pt'
        ),map_location=device))

X_test, y_t1_test, y_t2_test, y_t3_test =     xgb_m3.extract_embeddings_and_labels(model, test_loader, device)

xgb_t1 = joblib.load('/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/xgboost_bg3_v3_t1_t3/xgb_t1.pkl')
pred_t1 = xgb_t1.predict(X_test)
#X_test_emb  = extract_embeddings(model, test_loader,  device)

/home/andreoseo/miniconda3/lib/python3.13/site-packages/xgboost/core.py:751: UserWarning: [11:30:23] WARNING: /__w/xgboost/xgboost/src/common/error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [9]:
model.load_state_dict(torch.load(os.path.join(
            '/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/bg3_deduplicatedn_t3_32',
            'model_weights.pt'
        ),map_location=device))

X_test, y_t1_test, y_t2_test, y_t3_test =     xgb_m3.extract_embeddings_and_labels(model, test_loader, device)
xgb_t3 = joblib.load('/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/xgboost_bg3_v3_t1_t3/xgb_t3.pkl')
pred_t3 = xgb_t3.predict(X_test)

In [10]:
model.load_state_dict(torch.load(os.path.join(
            '/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/bg3_v3_only_t2',
            'model_weights.pt'
        ),map_location=device))

X_test, y_t1_test, y_t2_test, y_t3_test =     xgb_m3.extract_embeddings_and_labels(model, test_loader, device)

# cargar t2 como dict de clasificadores individuales
clfs_t2 = {
    col: joblib.load(f'/home/andreoseo/Documents/MiSongGyny2026/models/checkpoints_true/xgboost_bg3_v3_t1_t3/xgb_t2_{col}.pkl')
    for col in cfg.columns.LABEL_COLS_T2
}


pred_t2 = np.column_stack([
    clfs_t2[col].predict(X_test)
    for col in cfg.columns.LABEL_COLS_T2
])

In [11]:
print(f"""
    pred1_shape -> {pred_t1.shape}
    pred2_shape -> {pred_t2.shape}
    pred3_shape -> {pred_t3.shape}
    """)

[2026-05-05 11:31] 
    pred1_shape -> (576,)
    pred2_shape -> (576, 4)
    pred3_shape -> (576,)
    


In [12]:
pred_t1 #1 -> M & 0 -> NM
pred_t2 #[0,0,0,0] -> acorde cfg -> ['type_sexualization', 'type_violence', 'type_hate', 'nothing']
pred_t3 #1 -> Y & 0 -> N

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0,
       0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 0, 1,
       1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1,
       1, 0, 0, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0,

In [13]:
# mapeos inversos
map_t1 = {1: 'M', 0: 'NM'}
map_t3 = {1: 'Y', 0: 'N'}

results_df = df.copy()
results_df['pred_t1'] = [map_t1[p] for p in pred_t1]
results_df['pred_t3'] = [map_t3[p] for p in pred_t3]
for i, col in enumerate(cfg.columns.LABEL_COLS_T2):
    results_df['pred_t2_' + col] = pred_t2[:, i].astype(int)

In [14]:
results_df.to_csv("results_xgboost.csv")